# 02 — Missing Values

Companion to [`../../data_cleaning/missing_values.md`](../../data_cleaning/missing_values.md).

We demonstrate detection, the indicator trick, and three imputation strategies on a synthetic dataset.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

rng = np.random.default_rng(0)
n = 500
df = pd.DataFrame({
    "age":    rng.normal(40, 12, n).round(),
    "income": rng.lognormal(10.5, 0.6, n).round(2),
    "spend":  rng.exponential(100, n).round(2),
})
# Make `income` MAR: missingness depends on age
miss_mask = rng.random(n) < (df['age'] / 200)
df.loc[miss_mask, 'income'] = np.nan
df.isna().mean()

## 1. Detect

In [ ]:
print(df.isna().sum())
sns.heatmap(df.isna().astype(int), cbar=False); plt.title('NaN map'); plt.show()

## 2. Indicator trick

In [ ]:
df['income_was_missing'] = df['income'].isna().astype(int)
df.head()

## 3. Three strategies

In [ ]:
baseline = SimpleImputer(strategy='median')
knn      = KNNImputer(n_neighbors=5)
mice     = IterativeImputer(random_state=0, max_iter=10)

X = df[['age', 'income', 'spend']]
out = pd.DataFrame({
    'median': baseline.fit_transform(X)[:, 1],
    'knn':    knn.fit_transform(X)[:, 1],
    'mice':   mice.fit_transform(X)[:, 1],
})
out.describe()

## 4. Comparing the imputed distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True, sharey=True)
for ax, col in zip(axes, ['median', 'knn', 'mice']):
    sns.kdeplot(out[col], ax=ax, label='imputed')
    sns.kdeplot(df['income'].dropna(), ax=ax, label='observed')
    ax.set_title(col); ax.legend()
plt.tight_layout(); plt.show()

## Takeaways

- Median imputation is a fine baseline but shrinks variance.
- KNN respects local structure but is slow on big data.
- MICE / iterative imputation is the most faithful in the MAR setting.
- In all cases, keep the `was_missing` indicator.